# Breaking the Blend - template for a new galaxy

A clean version of the automated Breaking the Blend workflow (Veenema et al. 2026) **with no galaxy-specific
settings**. Use it for any NIRSpec and/or MIRI/MRS (or other IFS) data: fill in the cells marked **EDIT**,
run, inspect, and refine. Defaults are generic starting points, not tuned values.

| Section | What happens |
|---|---|
| 0 | **EDIT** - name, data folder, redshift, run and figure switches |
| 1 | Find & load cubes, check the AGN position |
| 2 | **EDIT** - emission lines and (optional) per-line settings |
| 3 | **Step 1** - line maps and the dispersion of each component |
| 4 | **EDIT** - **Step 2** - kinematic models: which parameters are fixed / fitted, which lines trace what |
| 5 | **EDIT** - **Step 3** - decomposition: where the model parameters and dispersions come from |
| 6 | **Step 4** - stacked AGN outflow fraction map |
| 7 | **Step 5** - AGN- and star-formation-dominated spectra |

Every slow step is saved in `RESULTS_DIR/GALAXY/` and re-loaded next time (recomputed automatically when its settings
change). Work through the sections in order the first time and look at every figure - the checks are what make the
result trustworthy. See `01_NGC5728.ipynb` for a fully worked example.

## 0. Settings - EDIT

In [ ]:
# =====================================================================
# 0. ANALYSIS SETTINGS
# =====================================================================
GALAXY       = "MyGalaxy"              # analysis name -> results are saved in RESULTS_DIR/GALAXY/
DATA_DIR     = "path/to/cubes"         # folder containing the *_s3d.fits(.gz) cubes (searched recursively)
RESULTS_DIR  = "results"
Z            = None                    # redshift  <-- REQUIRED
DISTANCE_MPC = None                    # optional distance [Mpc] for the physical scale (None = from Z)

# Re-run switches: False -> load saved results if they exist (and their settings are unchanged)
RERUN_LINE_FITS     = False
RERUN_KINEMATICS    = False
RERUN_DECOMPOSITION = False

# Kinematic fitting. "least_squares" is fast for a first pass; switch to "emcee" for final results
# (posteriors / uncertainties), e.g. dict(method="emcee", nwalkers=32, nsteps=20000, burnin=5000, sigma_v="residual")
MCMC = dict(method="least_squares", n_starts=20, sigma_v="residual")

# Figure output: "png", "pdf", "both" or "none"  (saved in RESULTS_DIR/GALAXY/figures/)
SAVE_FIGURES = "both"
FIGURE_DPI   = 200

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
try:
    import breaking_the_blend as btb
except ModuleNotFoundError:   # not installed (pip install -e .): use the copy in this repository
    import sys, os
    sys.path.insert(0, os.path.abspath(".."))
    import breaking_the_blend as btb
from breaking_the_blend import plotting as bplt     # also sets the publication plotting style

if Z is None:
    raise ValueError("Set the redshift Z in Section 0")
proj = btb.Project(GALAXY, data_dir=DATA_DIR, z=Z, distance_mpc=DISTANCE_MPC, results_dir=RESULTS_DIR,
                   save_figures=SAVE_FIGURES, figure_dpi=FIGURE_DPI)

## 1. Data: find the cubes and locate the AGN

All `*s3d.fits*` files under `DATA_DIR` are found and named by band (`ch3-short`, `g395h-f290lp`, ...). Use
`exclude=[...]` to skip files (e.g. background cubes) or `proj.set_cube_files({name: path})` to choose them by hand.

The AGN position of each cube is its (centroided) continuum peak. **Check every cross** in the overview figure; if
one is off the nucleus (e.g. a bright star-forming knot, or a weak AGN), give its pixel position in `AGN_OVERRIDES`.

In [ ]:
files = proj.find_cubes(exclude=["*bkg*"])

In [ ]:
AGN_OVERRIDES = {}            # EDIT if needed: {"cube name": (x, y)} in pixels
PIXEL_SCALE_OVERRIDES = {}    # normally empty: the header pixel size is used

LOAD_ERR = True               # also read the ERR extensions (spectrum errors in Step 5; doubles cube memory)

cubes = proj.load_cubes(agn_overrides=AGN_OVERRIDES, pixel_scale_overrides=PIXEL_SCALE_OVERRIDES, load_err=LOAD_ERR)
fig = bplt.plot_cubes_overview(cubes);  proj.savefig(fig, "cubes_overview")

## 2. Emission lines - EDIT

Start from a broad list; lines not covered by any loaded cube are skipped automatically. After Step 1, **remove lines
that are weak or blended** in your data. Per-line settings (`LINE_SETTINGS`) are only needed where the defaults
fail - look at the spaxel fits in Step 1 first, or test settings quickly with `03_check_line_settings.ipynb`. Useful settings: `line_region` (rest $\mu$m range of the line;
widen it for broad outflows), `fit_window`, `mask_percentile`, `p0_sigma_kms`, `p0_dv_kms`, `p0_amp_frac`, `cube`.
All options: `help(btb.LineSpec)`; the catalogue: `btb.LINE_CATALOGUE`.

In [ ]:
LINES = ["FeII", "ArII", "NeII", "SIII", "ArIII", "SIV", "NeIII", "OIV", "ArVI", "MgIV", "NeV", "NeVI"]

COMMON_SETTINGS = dict(mask_percentile=40)          # applied to every line (40 -> brightest 60% of spaxels)
LINE_SETTINGS = {
    # [Cl II] 14.368 um lies ~1000 km/s from [Ne V] 14.32 um: keep it out of the [Ne V] window
    "NeV": dict(line_region=(14.295, 14.345), fit_window=(14.265, 14.350)),
}
# Example of a full set of per-line settings (those used for NGC 5728 in notebook 01). Uncomment and adapt
# lines as needed - check them first with 03_check_line_settings.ipynb.
# LINE_SETTINGS = {
#     "FeII":  dict(cube="ch1-short",  line_region=(5.320, 5.355),  mask_percentile=55,
#                   p0_amp_frac=(0.67, 0.33), p0_dv_kms=(-2.8, 1.1),  p0_sigma_kms=(112, 225)),
#     "ArII":  dict(cube="ch1-long",   line_region=(6.975, 7.000),  mask_percentile=50,
#                   p0_amp_frac=(0.77, 0.50), p0_dv_kms=(-21.5, 4.3), p0_sigma_kms=(86, 215)),
#     "NeII":  dict(cube="ch3-short",  line_region=(12.800, 12.830), mask_percentile=70,
#                   p0_amp_frac=(0.50, 0.33), p0_dv_kms=(-23.4, 2.3), p0_sigma_kms=(105, 164)),
#     "SIII":  dict(cube="ch4-short",  line_region=(18.650, 18.780), mask_percentile=55,
#                   p0_amp_frac=(0.71, 0.40), p0_dv_kms=(-2.4, 3.2),  p0_sigma_kms=(112, 152)),
#     "ArIII": dict(cube="ch2-medium", line_region=(8.980, 9.015),  mask_percentile=55,
#                   p0_amp_frac=(0.71, 0.43), p0_dv_kms=(-1.7, 1.7),  p0_sigma_kms=(100, 200)),
#     "SIV":   dict(cube="ch2-long",   line_region=(10.485, 10.535), mask_percentile=55,
#                   p0_amp_frac=(0.67, 0.33), p0_dv_kms=(0.0, 28.5),  p0_sigma_kms=(86, 257)),
#     "NeIII": dict(cube="ch3-long",   line_region=(15.520, 15.600), mask_percentile=60,
#                   p0_amp_frac=(0.77, 0.33), p0_dv_kms=(-3.9, 3.9),  p0_sigma_kms=(106, 173)),
#     "OIV":   dict(cube="ch4-long",   line_region=(25.830, 25.970), mask_percentile=55,
#                   p0_amp_frac=(0.67, 0.33), p0_dv_kms=(0.0, 1.2),   p0_sigma_kms=(81, 127)),
#     "ArVI":  dict(cube="g395h-f290lp", line_region=(4.524, 4.537), mask_percentile=55,
#                   p0_amp_frac=(0.33, 0.50), p0_dv_kms=(-3.3, 3.3),  p0_sigma_kms=(66, 165)),
#     "MgIV":  dict(cube="g395h-f290lp", line_region=(4.481, 4.492), mask_percentile=55,
#                   p0_amp_frac=(0.33, 0.67), p0_dv_kms=(-0.7, 0.1),  p0_sigma_kms=(134, 267)),
#     "NeV":   dict(cube="ch3-medium", line_region=(14.295, 14.345), fit_window=(14.265, 14.350),  # avoid [Cl II]
#                   mask_percentile=50),
#     "NeVI":  dict(cube="ch2-short",  line_region=(7.645, 7.670),  mask_percentile=50,
#                   p0_amp_frac=(0.33, 0.67), p0_dv_kms=(-0.4, 0.0),  p0_sigma_kms=(78, 176)),
# }
# Note: `cube` names must match your loaded cubes, and settings tuned to one galaxy's line widths
# (line_region, p0_sigma_kms) may not suit another.

lines = btb.get_lines(LINES)
lines = {n: l.update(**{**COMMON_SETTINGS, **LINE_SETTINGS.get(n, {})}) for n, l in lines.items()}
lines = btb.assign_cubes(lines, cubes, drop_missing=True)
proj.save_config(lines=lines)

## 3. Step 1 - measure every line in every spaxel

Continuum subtraction, line flux, masking, single- and double-Gaussian fits and moment maps for every line.
The defaults are conservative: bounded fits (`fitter="trf"`), unweighted (`weighting="none"`; use `"err"` if your cubes
have reliable ERR extensions), started at each spaxel's moment-1 velocity, and the flux percentile computed inside the
field of view.

In [ ]:
FIT_OPTIONS = dict(fitter="trf", weighting="none", p0_center="moment", percentile_over="fov", dilate=2)

line_maps = proj.checkpoint("step1_line_maps", btb.fit_lines, cubes, lines, **FIT_OPTIONS,
                            rerun=RERUN_LINE_FITS, settings=dict(lines=lines, **FIT_OPTIONS))

**Check the fits.** Change `CHECK_LINE` to look at each line in turn. If the double-Gaussian fits look wrong, adjust
that line in `LINE_SETTINGS` (Section 2); it is re-fitted automatically.

In [ ]:
CHECK_LINE = list(line_maps)[len(line_maps) // 2]       # EDIT to inspect other lines
fig = bplt.plot_spaxel_fits(line_maps[CHECK_LINE], n=6);  proj.savefig(fig, f"spaxel_fits_{CHECK_LINE}")

In [ ]:
fig = bplt.plot_line_maps(line_maps, "flux");      proj.savefig(fig, "flux_maps")
fig = bplt.plot_line_maps(line_maps, "velocity");  proj.savefig(fig, "velocity_maps")
fig = bplt.plot_line_maps(line_maps, "sigma");     proj.savefig(fig, "single_gaussian_dispersion_maps")

Use these maps (ordered by IP) to decide, for Section 4, **which lines are dominated by the disc, which by the outflow,
and which show both**. Note the velocity colour range (`vlim=(-v, v)`) can be changed in every map function.

In [ ]:
fig = bplt.plot_line_quantities(line_maps, ("v_n", "v_b"));          proj.savefig(fig, "double_gaussian_velocity_maps")
fig = bplt.plot_line_quantities(line_maps, ("sigma_n", "sigma_b"));  proj.savefig(fig, "double_gaussian_dispersion_maps")
fig = bplt.plot_line_quantities(line_maps, ("mom0", "mom1", "mom2")); proj.savefig(fig, "moment_maps")

### Dispersion of each component

With $\sigma_{\rm outflow} > \sigma_{\rm disc}$ enforced per spaxel, the medians are the default fixed dispersions for
Step 3. For lines with a single component the narrow/broad split is meaningless - check the histograms, and use
`SIGMA_OVERRIDES = {"line": (sigma_disc, sigma_outflow)}` where needed.

In [ ]:
SIGMA_OVERRIDES = {}

sigmas_measured = btb.dispersion_medians(line_maps, sigma_range=(15, 600), overrides=SIGMA_OVERRIDES)
fig = bplt.plot_dispersion_histograms(line_maps, sigmas_measured, xlim=None)
proj.savefig(fig, "dispersion_histograms")
proj.save_config(sigmas_measured=sigmas_measured)

## 4. Step 2 - kinematic models - EDIT

Built-in models (offsets from the AGN in **pc**, angles in degrees, PA **east of north**):

* **thin rotating disc** - `v_disc`, `R_disc`, `inc_disc`, `pa_disc` (PA of the *receding* side);
* **hollow bicone** - `v_out`, `R_out`, `inc_out`, `pa_out` (PA of the *redshifted* cone), `theta_in`, `theta_out`;
* **combined** - $v = \alpha v_{\rm disc} + (1-\alpha) v_{\rm outflow}$, plus `v_sys`.

In `MODEL_PARAMS` a **number fixes** a parameter and a **tuple `(lo, hi)` fits** it with a uniform prior. Any
combination works. **If the inclinations and position angles are known (literature, other tracers), fix them** - a
partial field of view rarely constrains the geometry well. Below everything is free with broad priors as a starting
point. If a fitted model looks mirrored, the PA is 180 deg off.

Keep the `(lo, hi)` ranges **wide** unless you have a reason to restrict them. A fit that ends on the edge of its range
is reported as `<-- at prior limit`: that value is set by the prior rather than the data, so widen that range (or fix
the parameter deliberately). Least-squares errors are approximate (and zero for the opening angles, whose effect is
not smooth); use `method="emcee"` for meaningful uncertainties.

In [ ]:
MODEL_PARAMS = {
    "v_sys":     0.0,           # km/s  (0 if the redshift is right; e.g. (-100, 100) to fit)
    # ---- disc ---------------------------------------------------------------------------------
    "inc_disc":  (10, 80),      # deg   EDIT: fix to a known value if available, e.g. 45.0
    "pa_disc":   (0, 360),      # deg E of N (receding side)
    "v_disc":    (30, 600),     # km/s
    "R_disc":    (20, 2000),    # pc
    # ---- outflow -----------------------------------------------------------------------------
    "inc_out":   (10, 80),      # deg
    "pa_out":    (0, 360),      # deg E of N (redshifted cone)
    "v_out":     (50, 2000),    # km/s
    "R_out":     (50, 3000),    # pc
    "theta_in":  (0, 60),       # deg
    "theta_out": (20, 120),     # deg
    # ---- combined -----------------------------------------------------------------------------
    "alpha":     (0, 1),
}

**Line groups - EDIT after looking at the Step 1 maps.** The initial guess below uses the ionisation potential:
low-IP lines (< 25 eV) for the disc, high-IP lines (> 55 eV) for the outflow, and every line below 90 eV for the
decomposition. Lines with **no** disc component (check the narrow-component maps) should be left out of
`DECOMPOSE_LINES` - they still constrain the outflow.

In [ ]:
DISC_LINES      = [n for n, l in lines.items() if l.ip < 25]     # EDIT
OUTFLOW_LINES   = [n for n, l in lines.items() if l.ip > 55]     # EDIT
DECOMPOSE_LINES = [n for n, l in lines.items() if l.ip < 90]     # EDIT
print("disc:", DISC_LINES, "\noutflow:", OUTFLOW_LINES, "\ndecompose:", DECOMPOSE_LINES)

RUN_KINEMATIC_FITS = True
# For the geometry schematics only: which side of the disc minor axis is nearer to us
# (+1: the side at PA_disc + 90 deg, -1: PA_disc - 90 deg) - set from dust lanes if known.
DISC_NEAR_SIDE = +1
# Geometry fitted in the single-component fits (4.1, 4.2) is averaged and FIXED in the combined fits (4.3),
# which otherwise have too much freedom. Set False to keep it free there too.
FIX_GEOMETRY_IN_COMBINED = True

In [ ]:
GEOM = ("inc_disc", "pa_disc", "inc_out", "pa_out")
free = lambda keys: [k for k in keys if not np.isscalar(MODEL_PARAMS[k])]
disc_fits, outflow_fits, combined_fits = {}, {}, {}

if RUN_KINEMATIC_FITS and DISC_LINES:
    # 4.1 disc-only fits to the disc-dominated lines
    disc_fits = proj.checkpoint("step2a_disc_fits", btb.fit_many, line_maps, DISC_LINES, "disc",
                                params=MODEL_PARAMS, **MCMC, rerun=RERUN_KINEMATICS,
                                settings=dict(lines=DISC_LINES, params=MODEL_PARAMS, **MCMC))
    n = DISC_LINES[0]
    fig = bplt.plot_kinematic_fit(disc_fits[n], line_maps[n]);  proj.savefig(fig, f"disc_fit_{n}")

In [ ]:
if RUN_KINEMATIC_FITS and OUTFLOW_LINES:
    # 4.2 outflow-only fits to the outflow-dominated lines
    outflow_fits = proj.checkpoint("step2b_outflow_fits", btb.fit_many, line_maps, OUTFLOW_LINES, "outflow",
                                   params=MODEL_PARAMS, **MCMC, rerun=RERUN_KINEMATICS,
                                   settings=dict(lines=OUTFLOW_LINES, params=MODEL_PARAMS, **MCMC))
    n = OUTFLOW_LINES[-1]
    fig = bplt.plot_kinematic_fit(outflow_fits[n], line_maps[n]);  proj.savefig(fig, f"outflow_fit_{n}")

In [ ]:
if RUN_KINEMATIC_FITS:
    # 4.3 combined fits to every decomposable line, started near the single-component results
    start = {**btb.mean_params(disc_fits, free(("v_disc", "R_disc") + GEOM[:2])),
             **btb.mean_params(outflow_fits, free(("v_out", "R_out", "theta_in", "theta_out") + GEOM[2:]))}
    COMBINED_PARAMS = dict(MODEL_PARAMS)
    if FIX_GEOMETRY_IN_COMBINED:
        COMBINED_PARAMS.update({k: start[k] for k in GEOM if k in start})
    init = {k: (0.7 * v, 1.3 * v + 1) for k, v in start.items() if k not in GEOM}
    combined_fits = proj.checkpoint("step2c_combined_fits", btb.fit_many, line_maps, DECOMPOSE_LINES,
                                    "combined", params=COMBINED_PARAMS, init=init, **MCMC, rerun=RERUN_KINEMATICS,
                                    settings=dict(lines=DECOMPOSE_LINES, params=COMBINED_PARAMS, init=init, **MCMC))
    btb.fits_table(combined_fits)
    n = DECOMPOSE_LINES[len(DECOMPOSE_LINES) // 2]
    fig = bplt.plot_kinematic_fit(combined_fits[n], line_maps[n]);  proj.savefig(fig, f"combined_fit_{n}")

Parameters against IP (box: 25-75th percentile, whiskers: 16-84th, line: median; least-squares fits show error bars).
Parameters that are roughly constant are averaged for the decomposition; any that change systematically with IP (in
NGC 5728: $R_{\rm out}$, $\alpha$) get a linear trend. **Check the plot** before trusting a trend.

In [ ]:
if RUN_KINEMATIC_FITS:
    fig = bplt.plot_params_vs_ip(combined_fits, ["R_out", "alpha", "v_disc", "v_out"], style="box")
    proj.savefig(fig, "params_vs_IP")
    FIXED_IN_FITS = {k: v for k, v in COMBINED_PARAMS.items() if np.isscalar(v)}          # fixed values
    FIT_MEANS = btb.mean_params(combined_fits, [k for k in COMBINED_PARAMS if k != "alpha"])  # fitted: averaged
    R_OUT_TREND = (btb.trend_with_ip(combined_fits, "R_out")
                   if (not np.isscalar(COMBINED_PARAMS["R_out"]) and len(combined_fits) > 2) else None)

**Best-fit geometry.** Schematic of the disc and bicone from the combined fits (averaged over the lines; cones drawn out
to 2 $R_{\rm out}$). Left: as seen by the observer - parts of the cones behind the disc are drawn as a wire-frame, parts in
front are shaded, yellow lines mark where the cones cross the disc plane. Right: a 3-D view with the line of sight.
Check that it is physically sensible (e.g. the cone that is redshifted points away from us). `DISC_NEAR_SIDE` flips
which side of the disc is nearer to us.

In [ ]:
if RUN_KINEMATIC_FITS:
    FIT_GEOMETRY = btb.mean_geometry(combined_fits)
    fig = btb.plot_geometry(FIT_GEOMETRY, disc_near_side=DISC_NEAR_SIDE, title=f"{GALAXY}: best-fit geometry")
    proj.savefig(fig, "geometry_best_fit")

### Optional: velocity / dispersion maps from other codes (3DBarolo, MOKA3D, ...)

Any component can use a map produced elsewhere instead of the built-in model. Give the AGN pixel position in that map,
its pixel size, and the systemic velocity to subtract if it holds absolute velocities. Keys: `"v_disc"`, `"v_out"`,
`"sigma_disc"`, `"sigma_out"`. Where an imported map is undefined the built-in model / fixed dispersion is used.

In [ ]:
USE_EXTERNAL_MAPS = False
EXTERNAL = {}

if USE_EXTERNAL_MAPS:
    pc_scale = next(iter(cubes.values())).pc_per_arcsec
    EXTERNAL["v_disc"] = btb.load_external_map("path/to/model_velocity.fits",   # EDIT
                                               agn_xy=(0.0, 0.0),                # EDIT: centre in that map
                                               pixel_scale_arcsec=0.2,           # EDIT (or read from the header)
                                               pc_per_arcsec=pc_scale, subtract=0.0)
    lm = line_maps[DECOMPOSE_LINES[0]]
    fig = bplt.plot_velocity_panels({"observed": lm.velocity_map(),
                                     "imported disc": np.where(lm.good, EXTERNAL["v_disc"].on_grid(lm), np.nan)},
                                    lm.extent_pc(), vlim=400)
    proj.savefig(fig, "external_map_check")

## 5. Step 3 - constrained decomposition - EDIT

`DECOMP_PARAMS` selects where each line's model parameters come from:

* `"fit_mean"` (default) - fitted geometry + averages of the combined fits, with $R_{\rm out}$ from its trend with IP
  (set `USE_R_OUT_TREND = False` to use the average instead);
* `"fit_per_line"` - each line's own combined fit (more freedom; check weak lines carefully);
* `"fixed_common"` - your own values in `FIXED_COMMON`, the same for every line;
* `"fixed_per_line"` - your own values per line in `FIXED_LINE_PARAMS` (radii in pc).

`DECOMP_SIGMAS`: `"measured"` (Step 1 medians) or `"fixed"` (`FIXED_SIGMAS`).

In [ ]:
DECOMP_PARAMS   = "fit_mean"     # "fit_mean" | "fit_per_line" | "fixed_common" | "fixed_per_line"
DECOMP_SIGMAS   = "measured"     # "measured" | "fixed"
USE_R_OUT_TREND = True

# Only used with the "fixed_*" options - EDIT with your own values
FIXED_COMMON = dict(v_sys=0.0, inc_disc=None, pa_disc=None, v_disc=None, R_disc=None,    # km/s, deg, pc
                    inc_out=None, pa_out=None, v_out=None, R_out=None, theta_in=None, theta_out=None)
FIXED_LINE_PARAMS = {}           # {"line": {"v_disc": ..., "R_disc": ..., ...}} - overrides FIXED_COMMON per line
FIXED_SIGMAS = {}                # {"line": (sigma_disc, sigma_outflow)} [km/s]

# Optional: decompose spectra re-extracted with other windows / continuum, e.g.
#   {"NeII": dict(window=(12.72, 12.92), line_region=(12.785, 12.835), cont_order=1)}
DECOMP_RECUT = {}

In [ ]:
dmaps = {n: line_maps[n] for n in DECOMPOSE_LINES}
if DECOMP_PARAMS == "fit_mean":
    rel = {"R_out": (lambda ip, a=R_OUT_TREND[0], b=R_OUT_TREND[1]: a * ip + b)} \
          if (USE_R_OUT_TREND and R_OUT_TREND is not None) else None
    line_parameters = btb.line_params(dmaps, {**FIXED_IN_FITS, **FIT_MEANS}, ip_relations=rel)
elif DECOMP_PARAMS == "fit_per_line":
    line_parameters = btb.line_params(dmaps, {}, fits=combined_fits)
elif DECOMP_PARAMS in ("fixed_common", "fixed_per_line"):
    per_line = FIXED_LINE_PARAMS if DECOMP_PARAMS == "fixed_per_line" else None
    line_parameters = btb.line_params(dmaps, FIXED_COMMON, per_line=per_line)
    missing = {n: [k for k, v in p.items() if v is None] for n, p in line_parameters.items()}
    if any(missing.values()):
        raise ValueError(f"Fill in FIXED_COMMON / FIXED_LINE_PARAMS first - missing: {missing}")
else:
    raise ValueError(DECOMP_PARAMS)

if DECOMP_SIGMAS == "fixed":
    sigmas = {n: {"disc": FIXED_SIGMAS[n][0], "outflow": FIXED_SIGMAS[n][1]} for n in DECOMPOSE_LINES}
else:
    sigmas = sigmas_measured

proj.save_config(decomp_params_source=DECOMP_PARAMS, decomp_sigmas_source=DECOMP_SIGMAS,
                 decomposition_params=line_parameters, decomposition_sigmas=sigmas)

In [ ]:
DECOMP_OPTIONS = dict(names=DECOMPOSE_LINES, sigma_mode="fixed", recut=DECOMP_RECUT or None)

decomps = proj.checkpoint("step3_decompositions", btb.decompose_lines, line_maps, line_parameters, sigmas,
                          cubes=cubes, external=EXTERNAL or None, **DECOMP_OPTIONS, rerun=RERUN_DECOMPOSITION,
                          settings=dict(params=line_parameters, sigmas=sigmas, external=sorted(EXTERNAL),
                                        **DECOMP_OPTIONS))

Disc-fraction map of every line, and the residual maps: **coherent residual structure** means the two-component
model does not describe that line (a third component, wrong velocities/dispersions, or a blend).

In [ ]:
fig = bplt.plot_fraction_maps(decomps, "f_disc");  proj.savefig(fig, "disc_fraction_maps")
fig = bplt.plot_residual_maps(decomps);            proj.savefig(fig, "residual_maps")

## 6. Step 4 - stack all lines into one map

Per-line $f_{\rm disc}$ maps on a common parsec grid, weighted by $w = 1 - 2|\langle f_{\rm disc}\rangle - 0.5|$, keeping
points covered by at least `min_lines` lines. `MAP_LIM_PC` / `SCALEBAR_PC` only affect the figure.

In [ ]:
STACK_OPTIONS = dict(scheme="linear", combine="mean", min_lines=2)
MAP_LIM_PC  = None       # e.g. 1500 to zoom the figure
SCALEBAR_PC = None       # e.g. 500 for a scale bar
SF_MAX, AGN_MIN = 0.2, 0.8      # f_out thresholds for disc- / outflow-dominated (also used for the spectra)

stack = btb.stack_fraction_maps(decomps, **STACK_OPTIONS)
print(f"line-to-line scatter in f_disc: {btb.line_to_line_scatter(stack):.2f}")
FRACTIONS = btb.fraction_summary(stack, sf_max=SF_MAX, agn_min=AGN_MIN)
proj.save_config(fraction_summary=FRACTIONS)

fig = bplt.plot_stacked_map(stack, which="f_out", lim_pc=MAP_LIM_PC, scalebar_pc=SCALEBAR_PC, title=GALAXY)
proj.savefig(fig, "outflow_fraction_map")
proj.save(stack, "step4_stacked_map")
proj.export_stacked_map(stack)

**Geometry used in the decomposition** (averaged over the decomposed lines). With per-line parameters this is their
mean; if imported maps replaced a component (Section 4), its drawn geometry is only indicative.

In [ ]:
USED_GEOMETRY = btb.mean_geometry(line_parameters)
fig = btb.plot_geometry(USED_GEOMETRY, disc_near_side=DISC_NEAR_SIDE, title=f"{GALAXY}: geometry used in the decomposition")
proj.savefig(fig, "geometry_used")

**Robustness**: the map should not depend strongly on the weighting or on any single line.

In [ ]:
alternatives = {"linear weights": stack,
                "equal weights": btb.stack_fraction_maps(decomps, **{**STACK_OPTIONS, "scheme": "equal"}, verbose=False),
                "median": btb.stack_fraction_maps(decomps, **{**STACK_OPTIONS, "combine": "median"}, verbose=False)}
for n in DECOMPOSE_LINES:                                       # leave each line out in turn
    alternatives[f"without {n}"] = btb.stack_fraction_maps(decomps, names=[m for m in DECOMPOSE_LINES if m != n],
                                                           **STACK_OPTIONS, verbose=False)
btb.compare_stacks(alternatives, "linear weights")

## 7. Step 5 - AGN and star-forming spectra

Spaxels with $f_{\rm out} > $ `AGN_MIN` give the AGN-dominated spectrum, $f_{\rm out} < $ `SF_MAX` the star-formation-
dominated one (`method="weighted"` weights every spaxel by its fractions instead). Check that both regions contain
spaxels in the extraction plot.

**Errors** (third column of the exported `.txt` files, 1$\sigma$ in Jy, shaded in the plot): `err_method="cube"` adds the
pipeline ERR values of the selected spaxels in quadrature - a lower limit, because neighbouring spaxels of a resampled
cube share noise; `"empirical"` measures the channel-to-channel scatter of each extracted spectrum (includes correlated
noise, inflated next to strong lines); `"max"` takes the larger. The band-to-band scale factors' uncertainty is not included.

In [ ]:
SPECTRA_OPTIONS = dict(method="threshold", agn_min=AGN_MIN, sf_max=SF_MAX, min_spaxels=3,
                       err_method="cube")   # errors: "cube" (ERR, quadrature), "empirical", "max" or None

spectra = btb.extract_all(cubes, stack, **SPECTRA_OPTIONS)
CHECK_BAND = lines[DECOMPOSE_LINES[0]].cube                    # EDIT to look at another band
fig = bplt.plot_extraction_regions(spectra[CHECK_BAND], cubes[CHECK_BAND]);  proj.savefig(fig, "extraction_regions")

In [ ]:
w_agn, f_agn, e_agn, _ = btb.stitch_spectra(spectra, "agn", with_errors=True)
w_sf,  f_sf,  e_sf,  _ = btb.stitch_spectra(spectra, "sf",  with_errors=True)
print(f"median S/N per channel: AGN {np.nanmedian(f_agn / e_agn):.0f}, SF {np.nanmedian(f_sf / e_sf):.0f}")

ERR_DISPLAY_SCALE = 1       # exaggerate the shaded errors for display only (e.g. 10 if they are invisible)
fig = bplt.plot_component_spectra(w_agn, f_agn, w_sf, f_sf, smooth=2, err_agn=e_agn, err_sf=e_sf,
                                  err_scale=ERR_DISPLAY_SCALE)
proj.savefig(fig, "component_spectra")
proj.save({"agn": (w_agn, f_agn, e_agn), "sf": (w_sf, f_sf, e_sf), "per_band": spectra}, "step5_spectra")
proj.export_spectra({"AGN": (w_agn, f_agn, e_agn), "SF": (w_sf, f_sf, e_sf)})   # wavelength, flux, error

In [ ]:
proj.summary()

## Checklist before trusting the result

* AGN crosses on the nucleus in every band (Section 1).
* Spaxel fits sensible for every line; weak / blended lines removed (Sections 2-3).
* Geometry: fixed to reliable values, or well constrained by the fits; models not mirrored (Section 4).
* Line groups chosen from the maps, not only from the IP guess (Section 4).
* Dispersion histograms show two components for every decomposed line (Section 3).
* Residual maps without coherent structure (Section 5).
* Stacked map stable when lines are left out or weights changed (Section 6).

The method assumes two kinematic components with $\sigma_{\rm outflow} > \sigma_{\rm disc}$, a rotating disc and a
(hollow) biconical outflow. For other geometries, replace a component with an imported map (Section 4) or extend
`breaking_the_blend.models`. Each step is a plain function, so individual lines can be re-done by hand and swapped into
`line_maps` / `decomps` before stacking (see the end of `01_NGC5728.ipynb`).

## 8. Aperture spectra: nuclear spectrum vs. the AGN-dominated spectrum

A **nuclear spectrum** is extracted in a circular aperture of 0.3" radius centred on the AGN, in every band, and
compared with the AGN-dominated spectrum from the decomposition (Step 5). More apertures can be added to
`APERTURES`, each with its own radius and position, given as an (east, north) offset from the AGN in arcsec, so the
aperture covers the same sky region in every band. The extraction uses exact fractional pixel weights (like photutils'
`method="exact"`), the same error method, detector-gap and field-of-view handling as Step 5, and the bands are stitched
with **the same band scale factors** as the AGN-dominated spectrum, so the two can be compared and subtracted directly.
A band is skipped if less than 90% of the aperture lies inside its observed field.

The aperture spectra are **not corrected for the point-spread function**: a fixed 0.3" aperture loses a growing fraction
of a point source's flux towards longer wavelengths (MIRI/MRS FWHM ~0.3" at 5 um, ~1" at 28 um). The band-to-band
rescaling hides the jumps between bands but not the slope this introduces within each band.

In [ ]:
APERTURES = {   # name: radius [arcsec] and (east, north) offset of the centre from the AGN [arcsec]
    "nuclear": dict(radius_arcsec=0.3, offset_arcsec=(0.0, 0.0)),
    # "my region": dict(radius_arcsec=0.5, offset_arcsec=(1.0, -0.5)),
}
APERTURE_METHOD     = "exact"           # "exact" (fractional pixel weights) or "center" (whole pixels)
APERTURE_ERR_METHOD = SPECTRA_OPTIONS.get("err_method", "cube")   # same as Step 5

ap_spectra = btb.extract_apertures(cubes, APERTURES, method=APERTURE_METHOD, err_method=APERTURE_ERR_METHOD)
# stitch with exactly the band scale factors of the AGN-dominated spectrum (Step 5), so the two are comparable
_, _, _, AGN_SCALES = btb.stitch_spectra(spectra, "agn", with_errors=True, verbose=False)
ap_stitched = {}
for name in APERTURES:
    print(f"--- {name}")
    w, f, e, _ = btb.stitch_spectra(ap_spectra[name], "flux", with_errors=True, scales=AGN_SCALES)
    ap_stitched[name] = (w, f, e)
proj.save({"apertures": APERTURES, "per_band": ap_spectra, "stitched": ap_stitched}, "step6_aperture_spectra")
proj.export_spectra(ap_stitched, key="aperture_spectrum")          # wavelength, flux, error

The apertures on the AGN outflow fraction map:

In [ ]:
PC_PER_ARCSEC = next(iter(cubes.values())).pc_per_arcsec
fig = bplt.plot_apertures_on_map(stack, APERTURES, PC_PER_ARCSEC, which="f_out", lim_pc=MAP_LIM_PC,
                                 title=f"{GALAXY}: apertures")
proj.savefig(fig, "apertures_on_outflow_map")

The aperture spectra next to the AGN-dominated spectrum (errors shaded):

In [ ]:
to_plot = {f"AGN-dominated (f_out > {AGN_MIN:g})": (w_agn, f_agn, e_agn)}
to_plot.update({f'{n} ({APERTURES[n]["radius_arcsec"]:g}" aperture)': ap_stitched[n] for n in APERTURES})
fig = bplt.plot_spectra(to_plot)
proj.savefig(fig, "aperture_vs_agn_dominated_spectrum")

### AGN-dominated spectrum minus the nuclear spectrum

The AGN-dominated region contains both the AGN point source and the outflow. Removing the nuclear aperture spectrum (taken to
represent the AGN point source) from it leaves the emission of the rest of the AGN-dominated region - mainly the outflow.
This is done **per band on the pixels themselves** (the aperture's pixel weights are removed from the region's), then
stitched with the AGN-dominated scale factors. The result equals *AGN-dominated - nuclear*, with errors that do not
double-count the pixels the two share; only the part of the aperture inside the region is removed (printed per band).

No matching scale is applied. `NUCLEAR_SCALE` (default 1) multiplies the nuclear contribution before removing it. A 0.3\"
aperture misses part of a point source's flux - increasingly towards long wavelengths (PSF FWHM ~0.3\" at 5 um, ~1\" at
28 um) - so some AGN light remains in the result. A PSF aperture correction (> 1) would remove it; a single number is only
an approximation, as the correction depends on wavelength.

In [ ]:
SUB_APERTURE  = "nuclear"
NUCLEAR_SCALE = 1.0      # factor applied to the aperture before removing it (1 = as extracted; > 1 e.g. a PSF correction)

res_bands = btb.subtract_aperture(cubes, spectra, ap_spectra[SUB_APERTURE], "agn", scale=NUCLEAR_SCALE)
w_res, f_res, e_res, _ = btb.stitch_spectra(res_bands, "flux", with_errors=True, scales=AGN_SCALES)
print(f"residual < 0 in {100 * np.nanmean(f_res < 0):.0f}% of channels")

w_n, f_n, e_n = ap_stitched[SUB_APERTURE]
lab_ap = f'{SUB_APERTURE} ({APERTURES[SUB_APERTURE]["radius_arcsec"]:g}" aperture)' + (f" x {NUCLEAR_SCALE:g}" if NUCLEAR_SCALE != 1 else "")
fig, axes = plt.subplots(2, 1, figsize=(11, 8.5), sharex=True, constrained_layout=True)
bplt.plot_spectra({f"AGN-dominated (f_out > {AGN_MIN:g})": (w_agn, f_agn, e_agn),
                   lab_ap: (w_n, NUCLEAR_SCALE * f_n, NUCLEAR_SCALE * e_n)}, colors={lab_ap: "k"}, ax=axes[0])
bplt.plot_residual_spectrum(w_res, f_res, e_res, label=f"AGN-dominated - {SUB_APERTURE}", ax=axes[1])
proj.savefig(fig, "agn_dominated_minus_aperture")
proj.export_spectra({f"AGN_minus_{SUB_APERTURE}": (w_res, f_res, e_res)}, key="aperture_spectrum")

## 9. PSF-corrected spectra: AGN point source, AGN outflow and star formation

The AGN is unresolved, so its light is spread by the point-spread function (PSF): a 0.3" aperture misses part of it
(increasingly towards long wavelengths), and its PSF wings leak into the surrounding spaxels - including the
AGN-dominated and SF-dominated regions. Treating the nuclear aperture of Section 8 as a point source at the AGN position,
the PSF model gives for every band and wavelength the fraction of the point-source flux inside the aperture (EE) and the
fractions that fall in the AGN- and SF-dominated regions (f_AGN, f_SF; same spaxels and weights as Step 5). Then:

* **AGN** (saved as `AGN-fixed`) = the total flux of the AGN point source - by default (`PSF_POINT_SOURCE = "fit"`)
  from fitting the spaxels around the nucleus with point source × PSF + a smooth extended component, at every
  wavelength; with `"aperture"`, aperture flux / EE;
* **AGN-outflow** = AGN-dominated region - f_AGN x AGN - the AGN-dominated spectrum with the point source removed;
* **SF** = SF-dominated region - f_SF x AGN - the SF-dominated spectrum with the AGN's PSF wings removed.

**What is subtracted.** By default (`PSF_SUBTRACT = "continuum"`) only a smooth continuum of the point source is removed
from the regions - the unresolved AGN (hot-dust) continuum - so all line, PAH and H$_2$ emission stays in the region
spectra and the subtraction adds almost no noise. **Errors.** The exported files have four columns: wavelength, flux,
the *random* error (propagated from the ERR cubes; use this for line fitting) and the *PSF-model systematic*
(`PSF_SYS` × the correction; smooth and correlated across channels - an uncertainty on the continuum level, not noise).
Expect few PAH / H$_2$ features in the AGN point-source spectrum: that emission is extended, not from the AGN itself.

**PSF model.** `PSF_MODEL = "webbpsf"` uses STPSF (formerly WebbPSF; either package works) monochromatic MIRI/MRS and
NIRSpec IFU PSFs - installed with the Breaking the Blend environment (`conda env update -f environment.yml --prune` for an existing
one); its data files (~90 MB) download automatically to `~/data/stpsf-data` the first time (or set `STPSF_PATH`;
see https://stpsf.readthedocs.io). Without STPSF the notebook falls back to
`"gaussian"`: FWHM = 0.033 λ + 0.106" (MRS; Argyriou et al. 2023). PSF models do not include the extra broadening from
cube building, so real cube PSFs are somewhat wider; `PSF_SYMMETRIC = True` azimuthally averages the PSF so its
orientation relative to the sky-aligned cubes does not matter.

**Caveat - extended emission around the nucleus.** The `"aperture"` mode assumes everything in the aperture is the point
source; compact extended emission (stars, narrow-line region, nuclear star formation) is then scaled by 1/EE and removed
from the regions. The `"fit"` mode separates the sharp PSF core from smoother emission and is much less affected, but
where the extended emission is about as compact as the PSF (long wavelengths, FWHM ~1") no method can separate them:
there `AGN-fixed` is an upper limit and `AGN-outflow` a lower limit. All six spectra (raw and corrected) are saved so
the effect of the correction can be judged.

In [ ]:
PSF_MODEL     = "webbpsf"     # "webbpsf" (STPSF / WebbPSF instrument PSFs) or "gaussian"
PSF_APERTURE  = "nuclear"     # the Section 8 aperture (AGN-raw spectrum; centre of the point source)
PSF_POINT_SOURCE = "fit"      # "fit": PSF + smooth extended emission fitted around the nucleus (recommended);
                              # "aperture": aperture flux / EE (assumes all light in the aperture is AGN)
PSF_FIT_RADIUS = None         # "fit": radius of the fitted region [arcsec] (default max(0.6", 2 x FWHM))
PSF_SUBTRACT  = "continuum"   # removed from the regions: "continuum" (smooth AGN continuum; lines, PAH and H2 kept)
                              # or "full" (per-channel point-source spectrum incl. its lines; noisier)
MAX_BAND_SCALE = 3.0          # region spectra: drop bands that would need rescaling by more than this when stitching
PSF_ANNULUS   = None          # e.g. (0.6, 0.9) [arcsec]: estimate the extended emission in this annulus
PSF_SYMMETRIC = True          # azimuthally average the PSF
PSF_N_WAVE    = 6             # PSF sample wavelengths per band (interpolated in between)
PSF_SYS       = 0.10          # PSF-model uncertainty, as a fraction of each correction
FWHM_SCALE    = 1.0           # "gaussian" only: scale the FWHM (e.g. 1.1 to mimic cube-building broadening)

kw = dict(n_wave=PSF_N_WAVE, symmetric=PSF_SYMMETRIC, fwhm_scale=FWHM_SCALE, psf_sys=PSF_SYS, annulus_arcsec=PSF_ANNULUS,
          point_source=PSF_POINT_SOURCE, fit_radius_arcsec=PSF_FIT_RADIUS, subtract=PSF_SUBTRACT)
try:
    psf_res, psf_frac = btb.psf_correct(cubes, spectra, ap_spectra[PSF_APERTURE], model=PSF_MODEL, **kw)
except ImportError as ex:
    print(f"{ex}\n-> falling back to the Gaussian PSF model")
    PSF_MODEL = "gaussian"
    psf_res, psf_frac = btb.psf_correct(cubes, spectra, ap_spectra[PSF_APERTURE], model=PSF_MODEL, **kw)
fig = bplt.plot_psf_fractions(psf_frac)
fig.axes[0].set_title(f"PSF model: {PSF_MODEL}")
proj.savefig(fig, "psf_fractions")

**Check 1 - PSF contamination maps.** On the native spaxel grid of the band covering each wavelength: top, the share of
the AGN point-source flux falling in each spaxel (the PSF model itself); bottom, the share of each spaxel's observed flux
that is AGN PSF light. Blue/red contours: AGN- and SF-dominated regions; white circle: the nuclear aperture. With the
instrument PSF (STPSF) the diffraction wings reach several arcsec, so the SF regions receive a little AGN light -
increasingly towards long wavelengths; a Gaussian PSF has almost no wings, so it shows none.

In [ ]:
CONTAM_WAVES = [3.0, 5.0, 12.0, 21.0]     # rest-frame um (continuum wavelengths)
pkw = dict(model=PSF_MODEL, symmetric=PSF_SYMMETRIC, fwhm_scale=FWHM_SCALE)
contam_maps = [btb.psf_contamination(cubes, psf_res, lam, **pkw) for lam in CONTAM_WAVES]
for m in contam_maps:
    if m is not None:
        sp = spectra.get(m["band"])
        share = lambda mask: np.nansum(np.where(mask, m["psf_pct"], 0)) if mask is not None else np.nan
        print(f"{m['lam']:5.1f} um ({m['band']}): AGN point source {1e3 * m['agn_flux']:.1f} mJy; of it "
              f"{share(sp.agn_mask):.1f}% falls in the AGN-dominated region and {share(sp.sf_mask):.3f}% in the "
              f"SF-dominated region; median contamination of SF spaxels "
              f"{np.nanmedian(m['contam_pct'][sp.sf_mask]) if sp.sf_mask is not None and sp.sf_mask.any() else np.nan:.3f}%")
fig = bplt.plot_psf_contamination(contam_maps, spectra, ap_spectra[PSF_APERTURE])
proj.savefig(fig, "psf_contamination_maps")

**Check 2 - is the nucleus a point source?** Curves of growth of the observed continuum around the AGN (solid) against
the model PSF (dashed), normalised at 0.3". If the data grow faster than the PSF, there is extended emission around the
nucleus, which the point-source correction treats as AGN light - the AGN spectrum is then over-estimated (by up to the
gap between the curves at large radii) and the AGN-outflow spectrum over-subtracted. Then consider `PSF_ANNULUS`.

In [ ]:
curves = [btb.growth_curve(cubes, lam, **pkw) for lam in CONTAM_WAVES]
fig = bplt.plot_growth_curves(curves)
fig.axes[0].set_title(f"Curves of growth around the AGN ({PSF_MODEL} PSF)")
proj.savefig(fig, "psf_growth_curves")
for c in curves:
    if c is not None:
        print(f"{c['lam']:5.1f} um ({c['band']}): flux within {c['radii'][-1]:g}\" = {c['data'][-1]:.2f}x the 0.3\" flux "
              f"(point source: {c['psf'][-1]:.2f}x)")

Seven spectra, saved as `spectrum_<label>.txt` (wavelength, flux, random error, PSF systematic): **AGN-raw** (nuclear
aperture, uncorrected), **AGN-fixed** (AGN point source per channel), **AGN-continuum** (its smooth continuum),
**AGN-dominated** (f_out > `AGN_MIN`, Step 5), **AGN-outflow** (AGN-dominated with the point source removed),
**SF-dominated** (f_out < `SF_MAX`, Step 5) and **SF** (SF-dominated with the AGN's PSF wings removed). Region spectra
use the band scale factors of Step 5, dropping bands that would need more than `MAX_BAND_SCALE`. Right: corrected /
uncorrected.


In [ ]:
_, _, _, AGN_SCALES = btb.stitch_spectra(spectra, "agn", with_errors=True, verbose=False)
_, _, _, SF_SCALES = btb.stitch_spectra(spectra, "sf", with_errors=True, verbose=False)

def stitch_bands(bands, scales, name):
    """Stitch per-band spectra with fixed scale factors (None = absolute fluxes). Bands that would need rescaling by
    more than MAX_BAND_SCALE (barely covered by the region, e.g. NIRSpec for small SF regions) are dropped.
    Returns wave, flux, random error, systematic (PSF-model) error."""
    sc = {b: (1.0 if scales is None else scales.get(b, 1.0)) for b in bands}
    bad = [b for b, s in sc.items() if not (1 / MAX_BAND_SCALE <= s <= MAX_BAND_SCALE)]
    if bad:
        print(f"  {name}: dropping {', '.join(f'{b} (x{sc[b]:.1f})' for b in bad)}")
    keep = {b: sp_ for b, sp_ in bands.items() if b not in bad}
    w, f, e, _ = btb.stitch_spectra(keep, "flux", with_errors=True, scales={b: sc[b] for b in keep}, verbose=False)
    sys_bands = {b: btb.ApertureSpectrum(name, b, sp_.wave, getattr(sp_, "flux_sys", np.zeros_like(sp_.flux)), None,
                                          sp_.weights) for b, sp_ in keep.items()}
    ws, fs, _ = btb.stitch_spectra(sys_bands, "flux", scales={b: sc[b] for b in keep}, verbose=False)[:3]
    return w, f, e, np.interp(w, ws, fs)

def with_zero_sys(t):
    return (t[0], t[1], t[2], np.zeros_like(t[1]))

COMPARE = {
    "AGN-raw":       with_zero_sys(btb.stitch_spectra(ap_spectra[PSF_APERTURE], "flux", with_errors=True,
                                                       scales={b: 1.0 for b in ap_spectra[PSF_APERTURE]}, verbose=False)[:3]),
    "AGN-fixed":     stitch_bands(psf_res["AGN"], None, "AGN-fixed"),             # point source, per channel
    "AGN-continuum": stitch_bands(psf_res["AGN-continuum"], None, "AGN-continuum"),  # its smooth continuum
    "AGN-dominated": with_zero_sys((w_agn, f_agn, e_agn)),                          # f_out > AGN_MIN (Step 5)
    "AGN-outflow":   stitch_bands(psf_res["AGN-outflow"], AGN_SCALES, "AGN-outflow"),
    "SF-dominated":  with_zero_sys((w_sf, f_sf, e_sf)),                             # f_out < SF_MAX (Step 5)
    "SF":            stitch_bands(psf_res["SF"], SF_SCALES, "SF"),
}
PAIRS = [("AGN-raw", "AGN-fixed", "Nuclear 0.3\" aperture: raw vs AGN point source"),
         ("AGN-dominated", "AGN-outflow", f"AGN-dominated region: uncorrected vs point source removed ({PSF_SUBTRACT})"),
         ("SF-dominated", "SF", f"SF-dominated region: uncorrected vs AGN PSF wings removed ({PSF_SUBTRACT})")]
COL2 = {"AGN-raw": "0.55", "AGN-fixed": "k", "AGN-continuum": "orange", "AGN-dominated": "lightsteelblue",
        "AGN-outflow": "royalblue", "SF-dominated": "lightsalmon", "SF": "crimson"}
fig, axes = plt.subplots(3, 2, figsize=(15, 12), constrained_layout=True, gridspec_kw=dict(width_ratios=[2.3, 1]))
for row, (a, b, title) in enumerate(PAIRS):
    show = {a: COMPARE[a][:3], b: COMPARE[b][:3]}
    if row == 0:
        show["AGN-continuum"] = COMPARE["AGN-continuum"][:3]
    bplt.plot_spectra(show, colors=COL2, ax=axes[row, 0])
    axes[row, 0].set_title(title, fontsize=13)
    wa, fa = COMPARE[a][0], COMPARE[a][1]
    wb, fb = COMPARE[b][0], COMPARE[b][1]
    ax = axes[row, 1]; bplt.style_axes(ax)
    ax.plot(wb, fb / np.interp(wb, wa, fa), color=COL2[b], lw=1.2)
    ax.axhline(1, color="0.5", ls="--", lw=0.8)
    ax.set_ylabel(f"{b} / {a}"); ax.set_xlabel(r"Rest-frame wavelength [$\mu$m]")
fig.suptitle(f"PSF correction ({PSF_MODEL} PSF, point source: {PSF_POINT_SOURCE}, subtracted: {PSF_SUBTRACT})", fontsize=15)
proj.savefig(fig, "psf_corrected_spectra")
for k, (w, f, e, s) in COMPARE.items():
    neg = np.nanmean(f < 0)
    print(f"{k:>14s}: negative in {100 * neg:4.1f}% of channels; median S/N per channel {np.nanmedian(np.abs(f) / e):7.1f}; "
          f"PSF systematic {100 * np.nanmedian(s / np.abs(f)):4.1f}% of the flux")
for k, (w, f, e, s) in COMPARE.items():       # 4 columns: wavelength, flux, random error, PSF systematic
    np.savetxt(proj.file(f"spectrum_{k}", ".txt"), np.column_stack([w, f, e, s]),
               header="rest_wavelength_um  flux_Jy  err_random_Jy  err_psf_systematic_Jy")
    print(f"exported -> {proj.file(f'spectrum_{k}', '.txt')}")
proj.save({"spectra": COMPARE, "fractions": psf_frac, "settings": dict(model=PSF_MODEL, point_source=PSF_POINT_SOURCE,
           subtract=PSF_SUBTRACT, annulus=PSF_ANNULUS, fit_radius=PSF_FIT_RADIUS, symmetric=PSF_SYMMETRIC,
           psf_sys=PSF_SYS, max_band_scale=MAX_BAND_SCALE)}, "step7_psf_corrected")

## 10. Spaxel-by-spaxel extinction correction from the H$_2$ rotation diagram

The H$_2$ 0-0 S(3) line (9.665 $\mu$m) lies in the 9.8 $\mu$m silicate absorption, so dust in front of the warm
molecular gas pushes S(3) below the other rotational transitions in the excitation (rotation) diagram,
$\log(N_{\rm u}/g_{\rm u})$ against $E_{\rm u}/k$. Following Donnan et al. (2024, MNRAS 529, 1386; Sect. 4.3.2, Fig. 7),
a single foreground screen

$$F_{\rm obs}(\lambda) = F_{\rm int}(\lambda)\, e^{-\tau_{9.8}\,\tau(\lambda)}, \qquad \tau(9.8\,\mu{\rm m}) = 1,$$

is applied with increasing $\tau_{9.8}$ until S(3) is back in line with the other transitions. This is done in every spaxel:

1. **Line maps** - continuum-subtracted flux maps of S(1)-S(7) (all MRS), each in the band that covers it;
2. **Common grid** - every line convolved to the PSF at S(1) (the coarsest, 0.033 $\lambda$ + 0.106") and resampled
   (flux conserving) onto the MRS channel-3 grid of S(1);
3. **$\tau_{9.8}$ per spaxel** - `TAU_METHOD = "powerlaw"`: a power law $N_{\rm u}/g_{\rm u} \propto E_{\rm u}^{\,b}$ (straight line in
   log-log; a power-law distribution of gas temperatures) through the corrected transitions other than S(3), and the
   $\tau_{9.8}$ that puts the corrected S(3) on it; `"smoothness"`: the minimum of the Laplacian penalty
   $P_{\rm H_2} = \sum_i [\log(N_{i-1}/g_{i-1}) - 2\log(N_i/g_i) + \log(N_{i+1}/g_{i+1})]^2$ (Donnan et al. eq. 18). Both
   are solved exactly; uncertainties from Monte-Carlo draws of the line fluxes;
4. **$\tau_{9.8}$ map** on the channel-3 grid;
5. **Corrected spectra** - the $\tau_{9.8}$ map resampled onto every band, each spaxel multiplied by
   $e^{+\tau_{9.8}\tau(\lambda)}$, and the Step 5 extraction (the same $f_{\rm out} > $ `AGN_MIN` and $< $ `SF_MAX` spaxels) and
   stitching (the same band scale factors) repeated;
6. **Comparison** with the uncorrected Step 5 spectra.

**Extinction law.** Donnan et al. use the law of Donnan et al. (2023a): a power law plus the 9.8 and 18 $\mu$m silicate
absorption (with crystalline features), which is not tabulated in the paper. `EXT_LAW = "kvt"` uses the standard
power law + silicate law of PAHFIT (Smith et al. 2007): the Kemper, Vriend & Tielens (2004) profile with its 18 $\mu$m
extension, mixed 9:1 with a $\lambda^{-1.7}$ power law, normalised to 1 at 9.8 $\mu$m. Pass a `(wave, tau)` table to use another law.

**Caveats.** (i) The correction assumes everything in a spaxel (continuum, PAHs, ionised lines) is behind the same
screen as the warm H$_2$; Donnan et al. find that different components can see different extinction, so treat the
corrected spectra as "corrected for the H$_2$ extinction". (ii) The ortho-to-para ratio is assumed to be 3; a lower value
makes the diagram zig-zag (odd J high / even J low) and biases $\tau_{9.8}$ - look at the example diagrams. (iii) S(7)
is blended with [Mg VII] 5.503 $\mu$m in AGN: it is deblended with Gaussians, with the H$_2$ velocity and width tied to the
nearest unblended H$_2$ line; set `H2_FIT_LINES` without `"S7"` to leave it out. (iv) Negative $\tau_{9.8}$ (S(3) above the trend:
noise) are set to 0, and spaxels without a measurement get the value of the nearest measured spaxel (`TAU_FILL`).

In [ ]:
H2_LINES       = ["S1", "S2", "S3", "S4", "S5", "S6", "S7"]   # H2 0-0 S(J) transitions (all MRS)
H2_TARGET      = "S1"         # common grid: the band holding S(1) (MRS channel 3) - or a cube name, e.g. "ch3-long"
H2_MATCH_PSF   = True         # convolve every line to the PSF FWHM at S(1) before resampling
H2_HALFWIDTH   = 700          # half-width of the integrated line region [km/s] (continuum: 1200 km/s on each side)
H2_SNR_MIN     = 3            # transitions below this S/N are not used in a spaxel
TAU_METHOD     = "powerlaw"   # "powerlaw" (S(3) on the power law through the other lines) or "smoothness" (Donnan+24 eq. 18)
EXT_LAW        = "kvt"        # tau(lambda)/tau_9.8: "kvt" (PAHFIT power law + silicates) or a (wave, tau) table
H2_FIT_LINES   = None         # transitions used for tau_9.8 (None = all H2_LINES), e.g. without "S7"
TAU_SYS        = 0.05         # fractional flux uncertainty added in quadrature (calibration, PSF matching)
TAU_N_MC       = 200          # Monte-Carlo draws for the tau_9.8 uncertainty
N_EXAMPLES     = 5            # example spaxels (spanning the tau_9.8 range) ...
EXAMPLE_SPAXELS = None        # ... or your own: [(iy, ix), ...] on the H2_TARGET grid
TAU_FILL       = "nearest"    # spaxels without tau_9.8: "nearest", "median", "zero" (no correction) or a number
TAU_CLIP_NEG   = True         # negative tau_9.8 -> 0
TAU_SMOOTH     = 0.0          # Gaussian smoothing of the tau_9.8 map before applying it [arcsec FWHM]
TAU_MAX_ERR    = None         # treat spaxels with sigma(tau_9.8) above this as unmeasured (then filled)

fig = bplt.plot_extinction_law(EXT_LAW, H2_LINES)
proj.savefig(fig, "ext_law")

**Step 1 - H$_2$ line maps** on the native grid of each band (grey: S/N < `H2_SNR_MIN`; black contour: S/N = `H2_SNR_MIN`).

In [ ]:
h2_native = btb.h2_line_maps(cubes, H2_LINES, target=None, match_psf=False, halfwidth_kms=H2_HALFWIDTH)
fig = bplt.plot_h2_maps(h2_native, snr_min=H2_SNR_MIN)
proj.savefig(fig, "ext_h2_maps_native")

**Step 2 - common grid.** Every line PSF-matched to S(1) and resampled onto the channel-3 grid, where the
rotation diagram of each spaxel is built. The bands of S(5)-S(7) (channel 1) cover a smaller field: outside it a
spaxel only has S(1)-S(4), which is still enough (S(3) needs a transition on each side).

In [ ]:
h2_maps = btb.h2_line_maps(cubes, H2_LINES, target=H2_TARGET, match_psf=H2_MATCH_PSF, halfwidth_kms=H2_HALFWIDTH)
fig = bplt.plot_h2_maps(h2_maps, snr_min=H2_SNR_MIN)
proj.savefig(fig, "ext_h2_maps_common_grid")

**Step 3 - excitation diagrams.** Top: observed (black) and extinction-corrected (red) populations of example
spaxels spanning the $\tau_{9.8}$ range; red dashed: the power law through the corrected transitions other than S(3).
Bottom: the quantity minimised against $\tau_{9.8}$ (as Donnan et al. Fig. 7), with the best value (solid) and the
16-84 per cent range (dashed). The spaxels are numbered on the maps below.

In [ ]:
tau_map = btb.fit_tau(h2_maps, method=TAU_METHOD, law=EXT_LAW, snr_min=H2_SNR_MIN, sys_frac=TAU_SYS,
                      n_mc=TAU_N_MC, fit_lines=H2_FIT_LINES)
examples = EXAMPLE_SPAXELS or btb.example_spaxels(tau_map, N_EXAMPLES)
fig = bplt.plot_excitation_diagrams(tau_map, examples)
proj.savefig(fig, "ext_excitation_diagrams")
for j, (iy, ix) in enumerate(examples):
    print(f"#{j + 1} spaxel (iy, ix) = ({iy}, {ix}): tau_9.8 = {tau_map.tau[iy, ix]:.2f} "
          f"(+{tau_map.tau_hi[iy, ix] - tau_map.tau[iy, ix]:.2f} / -{tau_map.tau[iy, ix] - tau_map.tau_lo[iy, ix]:.2f}), "
          f"{tau_map.n_lines[iy, ix]} transitions, S(3) S/N = {tau_map.snr_s3[iy, ix]:.0f}")

**Step 4 - $\tau_{9.8}$ map** on the channel-3 grid, its uncertainty and the S/N of S(3) (grey: field without a
measurement). Contours: $f_{\rm out}$ = `SF_MAX` (orange) and `AGN_MIN` (blue) of the stacked map - the SF- and
AGN-dominated regions of Step 5. As a check, the other method is run too.

In [ ]:
fig = bplt.plot_tau_map(tau_map, examples, stack=stack, levels=(SF_MAX, AGN_MIN))
proj.savefig(fig, "ext_tau98_map")

other = "smoothness" if TAU_METHOD == "powerlaw" else "powerlaw"
tau_alt = btb.fit_tau(h2_maps, method=other, law=EXT_LAW, snr_min=H2_SNR_MIN, sys_frac=TAU_SYS, n_mc=20,
                      fit_lines=H2_FIT_LINES, verbose=False)
both = tau_map.ok & tau_alt.ok
if both.any():
    d = tau_alt.tau[both] - tau_map.tau[both]
    print(f"check - '{other}' minus '{TAU_METHOD}': median {np.median(d):+.2f}, scatter {1.4826 * np.median(np.abs(d - np.median(d))):.2f} "
          f"(median uncertainty of '{TAU_METHOD}': {np.nanmedian(tau_map.tau_err[both]):.2f})")
x_pc, y_pc = tau_map.grid.offsets_pc()
np.savez(proj.file("tau98_map", ".npz"), tau98=tau_map.tau, tau98_err=tau_map.tau_err, tau98_lo=tau_map.tau_lo,
         tau98_hi=tau_map.tau_hi, ok=tau_map.ok, x_pc=x_pc[0], y_pc=y_pc[:, 0], grid=tau_map.grid.name,
         pixel_scale=tau_map.grid.pixel_scale, method=TAU_METHOD)
proj.save({"tau_map": tau_map, "h2_maps": h2_maps, "examples": examples}, "step10_tau98")
print(f"saved -> {proj.file('tau98_map', '.npz')}")

**Step 5 - extinction-corrected AGN- and SF-dominated spectra.** The $\tau_{9.8}$ map is resampled onto every band
(area-averaged; gaps filled with `TAU_FILL`), each spaxel is corrected with $e^{+\tau_{9.8}\tau(\lambda)}$, and the spectra are
extracted with the Step 5 settings (same spaxels) and stitched with the Step 5 band scale factors, so any difference
from the Step 5 spectra is the extinction correction alone. Below: $\tau_{9.8}$ on three bands with the AGN (blue) and SF
(orange) spaxels outlined.

In [ ]:
EXT_OPTIONS = dict(SPECTRA_OPTIONS)        # as Step 5
spectra_ext, tau_on = btb.extinction_corrected_spectra(cubes, stack, tau_map, law=EXT_LAW, fill=TAU_FILL,
                                                       clip_negative=TAU_CLIP_NEG, smooth_fwhm_arcsec=TAU_SMOOTH,
                                                       max_err=TAU_MAX_ERR, **EXT_OPTIONS)
_, _, _, EXT_AGN_SCALES = btb.stitch_spectra(spectra, "agn", with_errors=True, verbose=False)   # Step 5 factors
_, _, _, EXT_SF_SCALES = btb.stitch_spectra(spectra, "sf", with_errors=True, verbose=False)
w_agn_x, f_agn_x, e_agn_x, _ = btb.stitch_spectra(spectra_ext, "agn", with_errors=True, scales=EXT_AGN_SCALES, verbose=False)
w_sf_x,  f_sf_x,  e_sf_x,  _ = btb.stitch_spectra(spectra_ext, "sf",  with_errors=True, scales=EXT_SF_SCALES, verbose=False)

bands = list(spectra_ext)
show = [bands[0], tau_map.grid.name if tau_map.grid.name in bands else bands[len(bands) // 2], bands[-1]]
vmax = np.nanpercentile(np.concatenate([tau_on[b].ravel() for b in show]), 99)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), constrained_layout=True)
for ax, b in zip(axes, show):
    bplt.style_axes(ax)
    sp_ = spectra_ext[b]
    t = np.where(sp_.footprint, tau_on[b], np.nan)
    im = ax.imshow(t, cmap="magma", vmin=0, vmax=vmax)
    for m, col in ((sp_.agn_mask, "deepskyblue"), (sp_.sf_mask, "darkorange")):
        if m is not None and m.any():
            ax.contour(m.astype(float), levels=[0.5], colors=col, linewidths=1.5)
    ax.plot(*cubes[b].agn_xy, "x", color="w", ms=10, mew=2)
    ax.set_title(f"{b} ({cubes[b].pixel_scale:.2f}\"/pix)")
    ax.set_xlabel("x [pixel]")
axes[0].set_ylabel("y [pixel]")
bplt.style_colorbar(plt.colorbar(im, ax=axes[-1]), r"$\tau_{9.8}$ applied")
proj.savefig(fig, "ext_tau98_on_bands")

**Step 6 - comparison.** Left: Step 5 (uncorrected) and extinction-corrected spectra of the AGN- and SF-dominated
regions; right: corrected / uncorrected, with (dashed) a single screen of the median $\tau_{9.8}$ of the region's spaxels.
The corrected spectra are saved as `spectrum_AGN-dominated_extcorr.txt` and `spectrum_SF-dominated_extcorr.txt`
(wavelength, flux, error).

In [ ]:
EXT_COMPARE = {
    "AGN-dominated":         (w_agn, f_agn, e_agn),
    "AGN-dominated_extcorr": (w_agn_x, f_agn_x, e_agn_x),
    "SF-dominated":          (w_sf, f_sf, e_sf),
    "SF-dominated_extcorr":  (w_sf_x, f_sf_x, e_sf_x),
}
COL3 = {"AGN-dominated": "lightsteelblue", "AGN-dominated_extcorr": "royalblue",
        "SF-dominated": "lightsalmon", "SF-dominated_extcorr": "crimson"}
ref_band = tau_map.grid.name if tau_map.grid.name in spectra_ext else list(spectra_ext)[-1]
fig, axes = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True, gridspec_kw=dict(width_ratios=[2.3, 1]))
for row, (a, comp, title) in enumerate([("AGN-dominated", "agn", f"AGN-dominated region (f_out > {AGN_MIN:g})"),
                                         ("SF-dominated", "sf", f"SF-dominated region (f_out < {SF_MAX:g})")]):
    b = f"{a}_extcorr"
    bplt.plot_spectra({a: EXT_COMPARE[a], b: EXT_COMPARE[b]}, colors=COL3, ax=axes[row, 0])
    axes[row, 0].set_title(title, fontsize=13)
    ax = axes[row, 1]; bplt.style_axes(ax)
    wa, fa = EXT_COMPARE[a][0], EXT_COMPARE[a][1]
    wb, fb = EXT_COMPARE[b][0], EXT_COMPARE[b][1]
    ax.plot(wb, fb / np.interp(wb, wa, fa), color=COL3[b], lw=1.2, label="corrected / uncorrected")
    m = getattr(spectra_ext[ref_band], f"{comp}_mask")
    if m is not None and m.any():
        t_med = float(np.median(tau_on[ref_band][m]))
        ax.plot(wb, btb.extinction_factor(wb, t_med, EXT_LAW), "k--", lw=1,
                label=rf"screen, median $\tau_{{9.8}}$ = {t_med:.2f}")
    ax.axhline(1, color="0.5", ls=":", lw=0.8)
    ax.set_ylabel("corrected / uncorrected"); ax.set_xlabel(r"Rest-frame wavelength [$\mu$m]")
    ax.legend(fontsize=10)
fig.suptitle(f"Extinction correction from H$_2$ ({TAU_METHOD} method, law: {EXT_LAW if isinstance(EXT_LAW, str) else 'custom'})",
             fontsize=15)
proj.savefig(fig, "ext_corrected_spectra")
for k in ("AGN-dominated_extcorr", "SF-dominated_extcorr"):
    w, f, e = EXT_COMPARE[k]
    np.savetxt(proj.file(f"spectrum_{k}", ".txt"), np.column_stack([w, f, e]),
               header="rest_wavelength_um  flux_Jy  err_Jy   (extinction corrected spaxel by spaxel, H2 tau_9.8)")
    print(f"exported -> {proj.file(f'spectrum_{k}', '.txt')}")
proj.save({"spectra": EXT_COMPARE, "per_band": spectra_ext, "settings": dict(method=TAU_METHOD, law=EXT_LAW,
           lines=H2_LINES, fit_lines=H2_FIT_LINES, target=H2_TARGET, fill=TAU_FILL, clip_negative=TAU_CLIP_NEG,
           smooth=TAU_SMOOTH, max_err=TAU_MAX_ERR, snr_min=H2_SNR_MIN)}, "step10_extinction_corrected")

## 11. Final spectra: per-spaxel extinction and PSF correction, band by band, then stitching

Section 9 removes the AGN's PSF wings from the region spectra band by band, but then stitches them with the scale
factors of the *uncorrected* Step 5 spectra. The PSF leakage differs from band to band (the PSF grows with wavelength
and every band has its own field and pixel grid), so the corrected bands no longer join and the stitched spectra jump
at the band edges. Here every correction is done per spaxel and per band first, and the bands are stitched only at the
end, with scale factors measured on the corrected spectra:

1. **Regions** - the AGN-dominated ($f_{\rm out} >$ `AGN_MIN`) and SF-dominated ($f_{\rm out} <$ `SF_MAX`) spaxels of Step 5, in every band;
2. **$\tau_{9.8}$ map** - from the H$_2$ maps of Section 10, refitted with the **Donnan et al. (2023a) extinction law**
   (`EXT_LAW_11`; the law used by Donnan et al. 2024: $0.1(9.8/\lambda)^{1.7} + 0.9\,\tau_{\rm Sil}$ with the empirical
   silicate profile of IRAS 08572+3915, crystalline features included), resampled onto every band;
3. **Spectra of the region spaxels** in each band, with each spaxel's extinction correction $e^{+\tau_{9.8,i}\tau(\lambda)}$, bands kept separate;
4. **PSF contamination per spaxel** - $P_i(\lambda)\,S(\lambda)$: the fraction of the AGN point source that the PSF model puts in
   spaxel $i$ (same PSF model and settings as Section 9), times the AGN point-source spectrum $S$ of that band (Section 9);
5. **Sum over the region spaxels** in each band: $F_R(\lambda) = \sum_i w_i\,[D_i(\lambda) - P_i(\lambda) S(\lambda)]\,e^{+\tau_{9.8,i}\tau(\lambda)}$;
6. **Stitching** from the short-wavelength end: the bluest usable band is kept as it is and every following band is scaled
   to match the spectrum built so far in their overlap (`STITCH_REF = "blue"`).

The PSF leakage is removed *before* the extinction correction: the leaked light is the AGN's own (spread by the
telescope), not light from behind the spaxel's H$_2$ screen. **AGN** is the 0.3" nuclear aperture spectrum (Section 8,
centred on the continuum peak) divided by the PSF encircled energy of the aperture in each band, then stitched in the
same way; the PSF wings removed from the regions use the point-source spectrum of Section 9 (`PSF_POINT_SOURCE`). The AGN
spectrum is not extinction-corrected.

**Coverage.** Each band sees the part of a region inside its own field. With `STITCH_REF = "blue"` the stitched spectrum
is normalised to the bluest band kept, so bands covering only a small corner of a region (typically NIRSpec or channel 1
for an extended SF ring) would set the normalisation; bands with less than `MIN_AREA_FRAC` of the largest region area
are therefore dropped (the sky area of each region per band is printed). The scale factors are printed: values far
from 1 mean the bands see different parts of the region.

In [ ]:
EXT_LAW_11    = "donnan2023"   # Donnan et al. (2023a) law (bundled); "kvt" = the law used in Section 10
TAU_METHOD_11 = TAU_METHOD     # as Section 10 ("powerlaw" or "smoothness")
STITCH_REF    = "blue"         # "blue": bluest usable band unscaled, then band by band to longer wavelengths;
                               # "largest": the band with the largest region area; or a band name
MIN_AREA_FRAC = 0.3            # drop bands whose region covers less than this fraction of the largest region area

# ---- Step 1: the regions (Step 5 spaxels) in every band
print(f"{'band':>14s}  {'AGN-dominated':>22s}  {'SF-dominated':>22s}")
for b, sp_ in spectra.items():
    na = int(sp_.agn_mask.sum()) if sp_.agn_mask is not None else 0
    ns = int(sp_.sf_mask.sum()) if sp_.sf_mask is not None else 0
    a2 = sp_.pixel_scale ** 2
    print(f"{b:>14s}  {na:5d} spx = {na * a2:6.2f} arcsec2  {ns:5d} spx = {ns * a2:6.2f} arcsec2")

# ---- Step 2: tau_9.8 map with the Donnan et al. (2023a) law, on every band
fig = bplt.plot_extinction_law(EXT_LAW_11, H2_LINES)
proj.savefig(fig, "s11_ext_law")
tau_map_11 = btb.fit_tau(h2_maps, method=TAU_METHOD_11, law=EXT_LAW_11, snr_min=H2_SNR_MIN, sys_frac=TAU_SYS,
                         n_mc=TAU_N_MC, fit_lines=H2_FIT_LINES)
both = tau_map_11.ok & tau_map.ok
if both.any():
    d = tau_map_11.tau[both] - tau_map.tau[both]
    print(f"tau_9.8 ({EXT_LAW_11}) minus tau_9.8 (Section 10, {EXT_LAW if isinstance(EXT_LAW, str) else 'custom'}): "
          f"median {np.median(d):+.2f}, 16-84%: {np.percentile(d, 16):+.2f} to {np.percentile(d, 84):+.2f}")
tau_on_11 = {b: btb.tau_on_cube(tau_map_11, cubes[b], fill=TAU_FILL, clip_negative=TAU_CLIP_NEG,
                                smooth_fwhm_arcsec=TAU_SMOOTH, max_err=TAU_MAX_ERR) for b in spectra if b in cubes}
fig = bplt.plot_tau_map(tau_map_11, stack=stack, levels=(SF_MAX, AGN_MIN))
fig.axes[0].set_title(f"law: {EXT_LAW_11}, method: {TAU_METHOD_11}", fontsize=12)
proj.savefig(fig, "s11_tau98_map")
np.savez(proj.file("tau98_map_s11", ".npz"), tau98=tau_map_11.tau, tau98_err=tau_map_11.tau_err, ok=tau_map_11.ok,
         x_pc=tau_map_11.grid.offsets_pc()[0][0], y_pc=tau_map_11.grid.offsets_pc()[1][:, 0], law=str(EXT_LAW_11),
         method=TAU_METHOD_11, grid=tau_map_11.grid.name)

**Steps 3-5 - corrected spectra per band** (not yet stitched). For each band: the leaked fraction of the AGN point
source in each region and the median $\tau_{9.8}$ of its spaxels. Below, the corrected spectra of every band in their own
colour, *before* stitching, so the band-to-band offsets that the stitching removes can be judged (left: AGN; middle:
AGN-outflow; right: SF).

In [ ]:
corr11 = btb.corrected_region_spectra(cubes, spectra, psf_res, tau_maps=tau_on_11, law=EXT_LAW_11, model=PSF_MODEL,
                                      n_wave=PSF_N_WAVE, symmetric=PSF_SYMMETRIC, fwhm_scale=FWHM_SCALE,
                                      subtract=PSF_SUBTRACT, psf_sys=PSF_SYS)

# AGN: 0.3" nuclear aperture / PSF encircled energy of the aperture, per band
agn_bands, agn_raw_bands = {}, {}
for b, ap_ in ap_spectra[PSF_APERTURE].items():
    if b not in psf_frac or not np.isfinite(ap_.flux).any():
        continue
    ee = np.interp(ap_.wave, psf_frac[b]["wave"], psf_frac[b]["ee"])
    s_ = btb.ApertureSpectrum("AGN", b, ap_.wave, ap_.flux / ee, None if ap_.flux_err is None else ap_.flux_err / ee,
                              ap_.weights, ap_.center_xy, ap_.radius_arcsec, ap_.pixel_scale, ap_.covered_frac, "psf")
    s_.flux_sys = PSF_SYS * np.abs(ap_.flux / ee - ap_.flux)
    agn_bands[b], agn_raw_bands[b] = s_, ap_

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), constrained_layout=True)
cmap_b = plt.get_cmap("turbo")
for ax, (title, bands_) in zip(axes, [("AGN (aperture / EE), per band", agn_bands),
                                      ("AGN-outflow corrected, per band", corr11["AGN-corr"]),
                                      ("SF corrected, per band", corr11["SF-corr"])]):
    bplt.style_axes(ax)
    names_ = sorted(bands_, key=lambda b: cubes[b].wave_range[0])
    allf = np.concatenate([np.asarray(bands_[b].flux, float) for b in names_]) if names_ else np.ones(1)
    for j, b in enumerate(names_):
        ax.plot(bands_[b].wave, bands_[b].flux, lw=0.8, color=cmap_b(j / max(len(names_) - 1, 1)), label=b)
    ax.set_yscale("log" if np.nanmean(allf <= 0) < 0.05 else "linear"); ax.set_title(title, fontsize=13)
    ax.legend(fontsize=8, ncol=2)
    ax.set_xlabel(r"Rest-frame wavelength [$\mu$m]")
axes[0].set_ylabel("Flux density [Jy]")
proj.savefig(fig, "s11_per_band_before_stitching")

**Step 6 - stitching and the final spectra.** Each spectrum is stitched from the short-wavelength end (`STITCH_REF`)
with factors measured on the spectrum itself. Left: the spectra; right: corrected / raw (the corrected spectra divided
by the raw ones; for the regions also the PSF-only and extinction-only corrections). Saved as `s11_spectrum_<label>.txt`
(wavelength, flux, random error, PSF systematic): **AGN-raw** (0.3" aperture), **AGN** (aperture / EE),
**AGN-dominated** and **SF-dominated** (raw, $f_{\rm out}$ > `AGN_MIN` / < `SF_MAX`), **AGN-outflow_corrected** and
**SF_corrected** (PSF leakage removed and extinction-corrected spaxel by spaxel).

In [ ]:
S11 = {}
for label, bands_, minfrac in [("AGN-raw", agn_raw_bands, 0.0), ("AGN", agn_bands, 0.0),
                               ("AGN-dominated", corr11["AGN-raw"], MIN_AREA_FRAC),
                               ("AGN-outflow_psf-only", corr11["AGN-psf"], MIN_AREA_FRAC),
                               ("AGN-dominated_ext-only", corr11["AGN-ext"], MIN_AREA_FRAC),
                               ("AGN-outflow_corrected", corr11["AGN-corr"], MIN_AREA_FRAC),
                               ("SF-dominated", corr11["SF-raw"], MIN_AREA_FRAC),
                               ("SF_psf-only", corr11["SF-psf"], MIN_AREA_FRAC),
                               ("SF-dominated_ext-only", corr11["SF-ext"], MIN_AREA_FRAC),
                               ("SF_corrected", corr11["SF-corr"], MIN_AREA_FRAC)]:
    if not bands_:
        print(f"{label}: no bands"); continue
    print(label)
    try:
        S11[label] = btb.stitch_bands(bands_, "flux", reference=STITCH_REF, min_area_frac=minfrac)
    except (ValueError, KeyError) as ex_:
        print(f"  not stitched: {ex_}")

COL11 = {"AGN-raw": "0.55", "AGN": "k", "AGN-dominated": "lightsteelblue", "AGN-outflow_corrected": "royalblue",
         "SF-dominated": "lightsalmon", "SF_corrected": "crimson"}
ROWS = [("AGN-raw", "AGN", None, None, "Nuclear 0.3\" aperture: raw vs PSF-corrected AGN"),
        ("AGN-dominated", "AGN-outflow_corrected", "AGN-outflow_psf-only", "AGN-dominated_ext-only",
         f"AGN-dominated region (f_out > {AGN_MIN:g}): raw vs PSF + extinction corrected"),
        ("SF-dominated", "SF_corrected", "SF_psf-only", "SF-dominated_ext-only",
         f"SF-dominated region (f_out < {SF_MAX:g}): raw vs PSF + extinction corrected")]
fig, axes = plt.subplots(3, 2, figsize=(15, 12.5), constrained_layout=True, gridspec_kw=dict(width_ratios=[2.3, 1]))
for row, (a, b, p_only, e_only, title) in enumerate(ROWS):
    if a not in S11 or b not in S11:
        continue
    ys = "log" if all(np.nanmean(S11[k_][1] <= 0) < 0.05 for k_ in (a, b)) else "linear"   # linear if partly negative
    bplt.plot_spectra({a: S11[a][:3], b: S11[b][:3]}, colors=COL11, ax=axes[row, 0], yscale=ys)
    axes[row, 0].set_title(title, fontsize=13)
    ax = axes[row, 1]; bplt.style_axes(ax)
    wa, fa = S11[a][0], S11[a][1]
    for k_, lab_, col_, ls_ in ((b, "corrected", COL11[b], "-"), (p_only, "PSF only", "0.4", "--"),
                                (e_only, "extinction only", "darkorange", ":")):
        if k_ is not None and k_ in S11:
            ax.plot(S11[k_][0], S11[k_][1] / np.interp(S11[k_][0], wa, fa), color=col_, ls=ls_, lw=1.1, label=lab_)
    ax.axhline(1, color="0.5", ls=":", lw=0.8)
    ax.set_ylabel(f"/ {a}"); ax.set_xlabel(r"Rest-frame wavelength [$\mu$m]")
    ax.legend(fontsize=10)
fig.suptitle(f"Section 11: PSF ({PSF_MODEL}, subtracted: {PSF_SUBTRACT}) + extinction ({EXT_LAW_11}) per spaxel, "
             f"stitched from the {STITCH_REF} end", fontsize=15)
proj.savefig(fig, "s11_final_spectra")

for k in ("AGN-raw", "AGN", "AGN-dominated", "AGN-outflow_corrected", "SF-dominated", "SF_corrected"):
    if k not in S11:
        continue
    w, f, e, s, sc, ref = S11[k]
    e = np.zeros_like(f) if e is None else e
    np.savetxt(proj.file(f"s11_spectrum_{k}", ".txt"), np.column_stack([w, f, e, s]),
               header=f"rest_wavelength_um  flux_Jy  err_random_Jy  err_psf_systematic_Jy   (stitched from {ref}; "
                      f"extinction law {EXT_LAW_11}, PSF {PSF_MODEL}/{PSF_SUBTRACT})")
    print(f"exported -> {proj.file(f's11_spectrum_{k}', '.txt')}")
proj.save({"spectra": S11, "per_band": corr11, "agn_per_band": agn_bands, "tau_map": tau_map_11,
           "settings": dict(law=EXT_LAW_11, method=TAU_METHOD_11, stitch_ref=STITCH_REF, min_area_frac=MIN_AREA_FRAC,
                            psf_model=PSF_MODEL, psf_subtract=PSF_SUBTRACT, psf_point_source=PSF_POINT_SOURCE,
                            tau_fill=TAU_FILL, clip_negative=TAU_CLIP_NEG, smooth=TAU_SMOOTH)}, "step11_final_spectra")